# 사전 설문, 우리 반은 어떨까?

**오늘 배우는 것**: 우리 반 사전 설문 응답을 직접 분석해서 막대그래프로 그려 봅니다.

코딩이 처음이라도 괜찮습니다. 셀 왼쪽의 ▶ 버튼만 차례로 누르면 됩니다. 막히면 손을 들어 주세요.

## 0단계 — 색은 왜 이 두 가지를 쓰나?

그래프 색은 **정보**입니다. 좋은 색은 두 가지를 만족해야 합니다.

1. **모두에게 보여야 합니다** — 색약(빨강·초록 구분이 어려움)인 분이 인구의 약 5~8%입니다.
2. **흑백 인쇄에서도 구분되어야 합니다** — 강의자료를 흑백으로 출력해도 의미가 살아야 합니다.

그래서 이 노트북은 일본의 Okabe & Ito 가 2008년 제안한 **Okabe-Ito 8색 팔레트**를 씁니다. Nature, PLOS 등 주요 학술지가 권장하는 표준입니다.

오늘 쓰는 두 색
- **Vermillion `#D55E00`** — 학습이 필요한 영역(전혀 모름)
- **Bluish Green `#009E73`** — 이미 능숙한 영역(보통 이상)

출처: <https://jfly.uni-koeln.de/color/>

## 1단계 — 한글 폰트 설치

그래프에서 한글이 □□□로 깨지지 않도록 도와주는 도구를 설치합니다. **처음 한 번만** 실행하면 됩니다.

In [ ]:
!pip install koreanize-matplotlib

## 2단계 — 분석 도구 가져오기

- `pandas` : 표(엑셀)를 다루는 도구
- `matplotlib` : 그래프를 그리는 도구
- `koreanize_matplotlib` : 한글 폰트를 자동으로 설정

셀을 실행해도 아무 출력이 없으면 **정상**입니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 한글 자동 설정

## 3단계 — 설문 데이터 파일 올리기

왼쪽 폴더 아이콘에 엑셀 파일을 **드래그**하거나, 아래 셀을 실행해 파일 선택창에서 고릅니다.

올릴 파일 이름: **`배포용_인공지능과 인문학 수강 사전 설문지xlsx.xlsx`**

In [ ]:
# Colab(코랩)에서만 사용. 본인 컴퓨터에서 돌릴 때는 이 셀을 건너뛰세요.
from google.colab import files
업로드 = files.upload()

## 4단계 — 데이터 읽기

엑셀 파일을 `pd.read_excel` 한 줄로 표(데이터프레임)로 만듭니다.
- 이 파일은 시트가 두 개라 어느 시트를 읽을지 `sheet_name` 으로 지정합니다.
- 마지막 줄 `df.head(3)` 은 표의 첫 3행을 미리보기로 보여 줍니다.

In [ ]:
파일이름 = "배포용_인공지능과 인문학 수강 사전 설문지xlsx.xlsx"
시트이름 = "설문지 응답 시트1"
df = pd.read_excel(파일이름, sheet_name=시트이름)
print("응답한 사람 수:", len(df), "명")
df.head(3)

## 5단계 — 어떤 질문이 있었나

표의 각 **열(컬럼)** 이 한 개의 설문 문항입니다.

In [ ]:
for i, c in enumerate(df.columns):
    print(i, c)

## 6단계 — 5점 척도 응답 분석

7번 문항(자기 수준 평가)은 5점 척도로 응답합니다.
- 전혀 모름 → 조금 앎 → 보통 → 잘 앎 → 매우 잘 앎

분석할 항목을 두 그룹으로 나눕니다.
- **사전 지식**: 파이썬 / 구글 코랩 / 한국어 말뭉치 — 처음부터 배워야 하는 내용
- **AI 도구 사용**: 프롬프트 / AI 글쓰기 / AI 이미지 생성 — 이미 익숙할 가능성

In [ ]:
사전지식_항목 = {
    "파이썬":        "7. 다음 항목에 대해 현재 자신의 수준을 표시해 주세요. [파이썬 등 프로그래밍 언어 사용]",
    "구글 코랩":      "7. 다음 항목에 대해 현재 자신의 수준을 표시해 주세요. [구글 코랩(Colab) 사용]",
    "한국어 말뭉치": "7. 다음 항목에 대해 현재 자신의 수준을 표시해 주세요. [말뭉치(코퍼스)의 개념과 활용]",
}
AI사용_항목 = {
    "프롬프트 작성":  "7.프롬프트 작성 (AI에게 효과적으로 질문하기",
    "AI 글쓰기":     "7. 다음 항목에 대해 현재 자신의 수준을 표시해 주세요. [AI를 활용한 글쓰기 또는 창작]",
    "AI 이미지 생성": "7. 다음 항목에 대해 현재 자신의 수준을 표시해 주세요. [AI를 활용한 이미지 생성]",
}

예시컬럼 = 사전지식_항목["파이썬"]
print(df[예시컬럼].value_counts())

## 7단계 — 비율 계산

각 항목별로 두 비율을 구합니다.
- **전혀 모름 비율** : 사전 지식 항목 — 학습이 필요한 사람의 몫
- **보통 이상 비율** : AI 사용 항목 — 이미 익숙한 사람의 몫

함수 `def`는 **계산 방법에 이름을 붙이는 일**입니다.

In [ ]:
def 전혀모름_비율(컬럼):
    응답 = df[컬럼].dropna()
    return round((응답 == "전혀 모름").sum() / len(응답) * 100)

def 보통이상_비율(컬럼):
    응답 = df[컬럼].dropna()
    return round(응답.isin(["보통", "잘 앎", "매우 잘 앎"]).sum() / len(응답) * 100)

사전지식_퍼센트 = {이름: 전혀모름_비율(컬럼) for 이름, 컬럼 in 사전지식_항목.items()}
AI사용_퍼센트   = {이름: 보통이상_비율(컬럼) for 이름, 컬럼 in AI사용_항목.items()}

print("사전 지식 — 전혀 모름 비율(%):", 사전지식_퍼센트)
print("AI 사용 — 보통 이상 비율(%):", AI사용_퍼센트)

## 8단계 — 막대그래프 그리기 (Okabe-Ito 색약 친화)

왼쪽 그래프: 사전 지식 — Vermillion `#D55E00` (학습 필요)
오른쪽 그래프: AI 도구 사용 — Bluish Green `#009E73` (이미 능숙)

두 색은 명도 차이가 충분해 흑백 인쇄에서도 구분됩니다.

In [ ]:
주황 = "#D55E00"   # Okabe-Ito Vermillion
청록 = "#009E73"   # Okabe-Ito Bluish Green

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

항목들 = list(사전지식_퍼센트.keys())[::-1]
값들   = [사전지식_퍼센트[k] for k in 항목들]
axes[0].barh(항목들, 값들, color=주황)
axes[0].set_xlim(0, 100)
axes[0].set_title("사전 지식 — 전혀 모름 비율 (%)")

항목들2 = list(AI사용_퍼센트.keys())[::-1]
값들2   = [AI사용_퍼센트[k] for k in 항목들2]
axes[1].barh(항목들2, 값들2, color=청록)
axes[1].set_xlim(0, 100)
axes[1].set_title("AI 사용 — 보통 이상 비율 (%)")

plt.tight_layout()
plt.show()

## 9단계 — 함께 생각해 볼 점

1. **격차의 의미** — 파이썬·코랩·말뭉치 같은 *도구* 는 모르지만 AI 활용은 절반 이상이 능숙합니다. 이 격차가 우리 수업에 어떤 영향을 줄까요?
2. **본인 위치** — 그래프 속 막대 안에 본인이 있다면 어디일까요?
3. **추가 질문** — 사전 설문에 빠진 질문이 있다면 무엇을 더 묻고 싶나요?
4. **색상 바꾸기** — 막대 색을 다른 Okabe-Ito 색(예: Sky Blue `#56B4E9`, Orange `#E69F00`) 으로 바꿔 보세요. 메시지가 달라지나요?

**오늘 배운 흐름**: 폰트 설치 → 도구 가져오기 → 데이터 읽기 → 컬럼 보기 → 빈도 세기 → 비율 계산 → 막대그래프